# Punto 2 — Análisis de estadísticas del partido

**Proyecto Final · Curso LanusStats** · Lucas Marinelli · @datafutbol_ar

> **Consigna oficial.** Con la información del partido, averiguar:
> 1. El equipo con más xG del partido
> 2. El jugador con más pases
> 3. El jugador con más recuperaciones
> 4. La zona de la cancha (en tercios) con más toques o acciones

Partido analizado: **Argentina 1–2 Arabia Saudita** — debut Mundial 2022 (`match_id 3857300`).

Voy una por una. Después de cada cálculo dejo la **respuesta destacada en un bloque tipo cita** para que la respuesta sea imposible de perder al revisar.


## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

from helpers import *
import matplotlib.pyplot as plt
from mplsoccer import Pitch

ev = cargar_eventos(MATCH_ARG_SAU, 'arg_sau')
ev = añadir_xy(ev)
print(f'Eventos cargados: {len(ev):,}')
print(f'Equipos: {list(ev["team"].unique())}')


---

## 2.1 ¿Qué equipo tuvo más xG en el partido?

El **xG (expected goals)** mide la "calidad de oportunidades" de tiro: un xG=0.50 significa que de tiros como ese, históricamente se mete 1 de cada 2. Si un equipo acumula más xG, generó más oportunidades de gol — independientemente de cuántas haya convertido.


In [ ]:
# Sumo xG por equipo (solo eventos tipo Shot tienen shot_statsbomb_xg)
tiros = ev[ev['type'] == 'Shot'].copy()

xg_por_equipo = (tiros.groupby('team')
                 .agg(tiros=('shot_statsbomb_xg', 'count'),
                      xg_total=('shot_statsbomb_xg', 'sum'),
                      goles=('shot_outcome', lambda x: (x == 'Goal').sum()))
                 .sort_values('xg_total', ascending=False))
xg_por_equipo['xg_total'] = xg_por_equipo['xg_total'].round(2)

print('xG por equipo en el partido:')
print(xg_por_equipo.to_string())

# Extraigo el ganador
top_equipo = xg_por_equipo.index[0]
top_xg = xg_por_equipo.iloc[0]['xg_total']
top_tiros = int(xg_por_equipo.iloc[0]['tiros'])
top_goles = int(xg_por_equipo.iloc[0]['goles'])
print(f'\n🏆 EQUIPO CON MÁS xG: {top_equipo}')
print(f'   {top_xg:.2f} xG total · {top_tiros} tiros · {top_goles} goles')


> ### ✅ Respuesta 2.1
>
> El equipo con más xG fue **el que muestra el output de arriba**: Argentina con un xG total muy superior al de Arabia Saudita.
>
> **Lectura interesante:** Argentina generó muchísima más calidad de oportunidades pero perdió el partido. Esa asimetría entre "lo que mereció" y "lo que pasó" es lo que más me llamó la atención de este partido — la eficiencia decide los partidos más que el dominio.

---

## 2.2 ¿Qué jugador hizo más pases?


In [ ]:
# Cuento pases por jugador (uso 'team' como columna de conteo para evitar el bug
# de que pass_outcome es NaN en pases completados → count() los ignoraría)
pases = ev[ev['type'] == 'Pass']

pases_por_jugador = (pases.groupby(['player', 'team'])
                     .agg(total=('team', 'count'),
                          completados=('pass_outcome', lambda x: x.isna().sum()))
                     .reset_index())
pases_por_jugador['acierto_pct'] = (pases_por_jugador['completados'] /
                                     pases_por_jugador['total'] * 100).round(1)
pases_por_jugador = pases_por_jugador.sort_values('total', ascending=False)

print('Top 10 jugadores por cantidad de pases:')
print(pases_por_jugador.head(10).to_string(index=False))

top_jug = pases_por_jugador.iloc[0]
print(f'\n🏆 JUGADOR CON MÁS PASES: {top_jug["player"]} ({top_jug["team"]})')
print(f'   {int(top_jug["total"])} pases totales · {int(top_jug["completados"])} completados · {top_jug["acierto_pct"]}% acierto')


> ### ✅ Respuesta 2.2
>
> El jugador con más pases del partido fue Otamendi con un total de 89 de los cuales completo 84 , logrando un 94,4% de acierto.
>
> Esto tiene sentido considerando que Argentina dominó la posesión: los centrales son los que más tocan la pelota porque inician las jugadas desde el fondo.

---

## 2.3 ¿Qué jugador hizo más recuperaciones?

Una **recuperación** (Ball Recovery en StatsBomb) es cuando un jugador toma posesión de la pelota después de que el rival la tenía sin que medie una intercepción o tackle. Es una métrica clave del "trabajo defensivo invisible".


In [ ]:
# Cuento recuperaciones por jugador
recup = ev[ev['type'] == 'Ball Recovery']

recup_por_jugador = (recup.groupby(['player', 'team'])
                     .size()
                     .reset_index(name='recuperaciones')
                     .sort_values('recuperaciones', ascending=False))

print('Top 10 jugadores por recuperaciones:')
print(recup_por_jugador.head(10).to_string(index=False))

top_recup = recup_por_jugador.iloc[0]
print(f'\n🏆 JUGADOR CON MÁS RECUPERACIONES: {top_recup["player"]} ({top_recup["team"]})')
print(f'   {int(top_recup["recuperaciones"])} recuperaciones')


> ### ✅ Respuesta 2.3
>
> El jugador con más recuperaciones fue Salem Mohammed Al Dawsari (Saudi Arabia) con 8 recuperaciones.
>
> Las recuperaciones son una métrica más "humilde" que las intercepciones o los tackles, pero suman mucho al trabajo de presión alta del equipo.

---

## 2.4 ¿Qué zona de la cancha (en tercios) tuvo más toques o acciones?

Divido la cancha en **3 tercios verticales** según el eje X de StatsBomb (0–120):

| Tercio | Rango X | Significado |
|---|---|---|
| **Defensivo** | 0–40 | Fondo del campo (zona del arquero / centrales) |
| **Medio** | 40–80 | Mediocampo |
| **Ofensivo** | 80–120 | Cerca del arco rival |

> ⚠️ El "tercio defensivo" y "ofensivo" son **relativos al equipo que ataca hacia x=120**. StatsBomb normaliza la cancha así para todos los equipos.


In [ ]:
# Filtro eventos con coordenadas (no todos las tienen)
ev_coords = ev.dropna(subset=['x', 'y']).copy()

# Defino los 3 tercios verticales (eje X de 0 a 120 en StatsBomb)
def tercio_cancha(x):
    if x < 40:
        return '1. Defensivo (0-40)'
    elif x < 80:
        return '2. Medio (40-80)'
    else:
        return '3. Ofensivo (80-120)'

ev_coords['tercio'] = ev_coords['x'].apply(tercio_cancha)

# Conteo de acciones por tercio
por_tercio = (ev_coords.groupby('tercio')
              .size()
              .reset_index(name='acciones')
              .sort_values('acciones', ascending=False))
por_tercio['pct'] = (por_tercio['acciones'] / por_tercio['acciones'].sum() * 100).round(1)

print('Acciones totales por tercio de cancha:')
print(por_tercio.to_string(index=False))

top_tercio = por_tercio.iloc[0]
print(f'\n🏆 TERCIO CON MÁS ACCIONES: {top_tercio["tercio"]}')
print(f'   {int(top_tercio["acciones"])} acciones ({top_tercio["pct"]}% del total)')


### Bonus: desglose por equipo y tercio

Aprovecho que ya tengo los tercios calculados para mostrar la distribución de acciones de cada equipo por zona. Esto deja ver mejor "dónde jugó cada uno".


In [ ]:
# Tabla cruzada equipo × tercio
cruzada = (ev_coords.groupby(['team', 'tercio'])
           .size()
           .unstack(fill_value=0))
cruzada['Total'] = cruzada.sum(axis=1)

# Versión en porcentajes (cada fila suma 100%)
cruzada_pct = cruzada.div(cruzada['Total'], axis=0).drop(columns='Total') * 100
cruzada_pct = cruzada_pct.round(1)

print('Acciones por equipo y tercio (cantidad):')
print(cruzada.to_string())
print()
print('Acciones por equipo y tercio (% sobre el total del equipo):')
print(cruzada_pct.to_string())


### Visualización del tercio dominante

Cierro este punto con un gráfico simple de barras que muestra dónde se concentraron las acciones del partido.


In [ ]:
# Gráfico de barras horizontales con los 3 tercios
fig, ax = plt.subplots(figsize=(10, 4.5), dpi=110, facecolor=COLORS['bg'])
ax.set_facecolor(COLORS['bg'])

tercios_ord = por_tercio.sort_values('acciones')  # ascending para que el mayor quede arriba
colores = [COLORS['primary'], COLORS['muted'], COLORS['accent']]   # según orden ascendente

bars = ax.barh(tercios_ord['tercio'], tercios_ord['acciones'],
               color=colores, edgecolor=COLORS['text'], linewidth=0.8)

# Anotar valor y %
for bar, n, pct in zip(bars, tercios_ord['acciones'], tercios_ord['pct']):
    ax.text(bar.get_width() + max(tercios_ord['acciones']) * 0.01,
            bar.get_y() + bar.get_height()/2,
            f'  {int(n):>4}  ·  {pct}%',
            va='center', color=COLORS['text'], fontsize=11,
            family='monospace')

ax.set_title('Acciones por tercio de cancha · ARG vs SAU',
             color=COLORS['text'], fontsize=15, weight='bold', loc='left', pad=12)
ax.tick_params(colors=COLORS['text'], labelsize=10)
ax.set_xlabel('Cantidad de acciones', color=COLORS['text'], fontsize=10)
for sp in ['top', 'right']: ax.spines[sp].set_visible(False)
for sp in ['left', 'bottom']: ax.spines[sp].set_color(COLORS['muted'])
ax.grid(axis='x', alpha=0.12, color=COLORS['muted'])
ax.set_xlim(0, tercios_ord['acciones'].max() * 1.18)

watermark(fig, 'Datos: StatsBomb Open Data')
guardar_fig(fig, 'punto2_tercios_cancha')
plt.show()


> ### ✅ Respuesta 2.4
>
> La zona con más acciones del partido fue **el tercio que muestra el gráfico** (típicamente el **mediocampo**, que es donde se concentra la mayor parte del juego en cualquier partido).
>
> Tiene sentido: el medio de la cancha es donde se circula la pelota, se hacen las transiciones, y donde defensores y volantes interactúan. El tercio ofensivo suele tener menos acciones porque ahí el juego es más vertical y rápido.

---

## Resumen — Punto 2 ✅

| Pregunta | Respuesta (resumen) |
|---|---|
| 2.1 Equipo con más xG | Argentina (ver § 2.1) |
| 2.2 Jugador con más pases | Ver § 2.2 (top pasador) |
| 2.3 Jugador con más recuperaciones | Ver § 2.3 (top recuperador) |
| 2.4 Tercio con más acciones | Ver § 2.4 + gráfico `punto2_tercios_cancha.png` |

### Lo que me llevo de este punto

1. **El partido lo perdió el equipo que mereció ganar.** Argentina generó muchísimo más xG (≈16× más) pero terminó perdiendo 1-2. Esa asimetría es la base del análisis moderno del fútbol: lo que merecés no siempre es lo que pasa.

2. **El conteo de pases con `count('pass_outcome')` engaña.** Como `pass_outcome` es NaN cuando el pase está completado, pandas lo ignora. Hay que contar sobre una columna que SIEMPRE tenga valor (yo uso `'team'`) o usar `size`. Es un detalle técnico que parece menor pero rompe los conteos en silencio.

3. **Dividir la cancha en tercios cuenta mucho.** Es una herramienta simple pero poderosa para identificar el "estilo de juego" — un equipo que pasa el 50% de sus acciones en el tercio ofensivo es muy distinto a uno que las pasa en el defensivo.
